# Notebook 6: Messages and Conversation Management

**What you'll learn:**
- How messages are structured (role + content blocks)
- All ContentBlock types (text, image, toolUse, toolResult)
- What the system prompt is and why it's separate from messages
- How conversation managers prevent context window overflow
- SlidingWindow vs Null conversation managers

**Prerequisite:** Complete [NB5_Hooks.ipynb](./NB5_Hooks.ipynb)

**Companion reading:** `06-messages-and-conversation.md`

---
## Message Format

Every message in the SDK is a dictionary with two keys:

```python
{
    "role": "user" or "assistant",
    "content": [ContentBlock, ContentBlock, ...]
}
```

**role** is who said it:
- `"user"` -- you (or tool results)
- `"assistant"` -- the AI model

**content** is a list of **ContentBlock** dictionaries. Each block has one key that determines its type:

| Type | Key | What it is | Example |
|------|-----|-----------|--------|
| Text | `"text"` | Plain text | `{"text": "Hello!"}` |
| Tool Use | `"toolUse"` | Model wants to call a tool | `{"toolUse": {"name": "calc", "input": {...}}}` |
| Tool Result | `"toolResult"` | Result from a tool | `{"toolResult": {"content": [{"text": "42"}]}}` |
| Image | `"image"` | An image | `{"image": {"format": "png", "source": {...}}}` |

**Source:** `src/strands/types/content.py`

In [ ]:
# ============================================================
# STEP 1: Call an agent with a tool and pretty-print messages
# ============================================================

import json
from strands import Agent, tool

@tool
def calculator(expression: str) -> str:
    """Perform a math calculation.
    
    Args:
        expression: A math expression like '2 + 2'.
    """
    return str(eval(expression))

agent = Agent(tools=[calculator], callback_handler=None)
result = agent("What is 15 * 7?")

# Pretty-print all messages
print(f"Total messages: {len(agent.messages)}")
print()

for i, msg in enumerate(agent.messages):
    print(f"=== Message [{i}] ===")
    print(f"Role: {msg['role']}")
    print(f"Content blocks: {len(msg['content'])}")
    
    for j, block in enumerate(msg['content']):
        # Determine the type of this content block
        if 'text' in block:
            print(f"  Block {j}: TEXT")
            print(f"    \"{block['text'][:100]}\"")
        elif 'toolUse' in block:
            print(f"  Block {j}: TOOL_USE")
            tu = block['toolUse']
            print(f"    name: {tu['name']}")
            print(f"    input: {json.dumps(tu['input'])}")
            print(f"    toolUseId: {tu['toolUseId']}")
        elif 'toolResult' in block:
            print(f"  Block {j}: TOOL_RESULT")
            tr = block['toolResult']
            print(f"    toolUseId: {tr.get('toolUseId')}")
            print(f"    status: {tr.get('status')}")
            for item in tr.get('content', []):
                if 'text' in item:
                    print(f"    result: \"{item['text'][:100]}\"")
    print()

In [ ]:
# ============================================================
# STEP 2: Build messages manually (understanding the format)
# ============================================================

# You can build messages yourself. This helps you understand
# the exact format the SDK uses internally.

# A user message with just text:
user_msg = {
    "role": "user",
    "content": [
        {"text": "What is 15 * 7?"}
    ]
}

# An assistant message with text AND a tool use request:
assistant_msg_with_tool = {
    "role": "assistant",
    "content": [
        {"text": "I'll calculate that for you."},
        {"toolUse": {
            "toolUseId": "abc123",    # Unique ID linking request to result
            "name": "calculator",      # Which tool to call
            "input": {                 # Arguments for the tool
                "expression": "15 * 7"
            }
        }}
    ]
}

# A user message with a tool result:
# (Tool results are sent as "user" messages so the model can read them)
tool_result_msg = {
    "role": "user",
    "content": [
        {"toolResult": {
            "toolUseId": "abc123",       # Must match the toolUse ID!
            "status": "success",
            "content": [
                {"text": "105"}
            ]
        }}
    ]
}

# An assistant message with the final answer:
final_msg = {
    "role": "assistant",
    "content": [
        {"text": "15 * 7 = 105"}
    ]
}

# This is exactly what agent.messages looks like after a tool call!
manual_messages = [user_msg, assistant_msg_with_tool, tool_result_msg, final_msg]

for i, msg in enumerate(manual_messages):
    types = [list(b.keys())[0] for b in msg['content']]
    print(f"[{i}] {msg['role']:10s} | blocks: {types}")

---
## System Prompt -- Separate From Messages

The **system prompt** is permanent instructions sent to the model with every call. It's NOT stored in `agent.messages`.

Think of it as the agent's **personality** or **job description**. It stays constant while messages grow.

```python
agent = Agent(system_prompt="You are a helpful math tutor.")
```

Every time the model is called, the SDK sends:
1. System prompt (separately)
2. All messages in `agent.messages`

In [ ]:
# ============================================================
# STEP 3: System prompt is NOT in messages
# ============================================================

# Create agent with a custom system prompt
agent_custom = Agent(
    system_prompt="You are a pirate. Always respond like a pirate.",
    callback_handler=None,
)

# Call the agent
result = agent_custom("What is 2+2?")
print(f"Result: {result}")
print()

# Check: is the system prompt in messages?
print("=== Messages ===")
for i, msg in enumerate(agent_custom.messages):
    text = ""
    for block in msg['content']:
        if 'text' in block:
            text = block['text'][:80]
    print(f"[{i}] {msg['role']:10s} | {text}")

print()
print("Notice: The system prompt is NOT in the messages list.")
print(f"System prompt: \"{agent_custom.system_prompt[:60]}\"")
print("It's sent separately to the model with every call.")

---
## Conversation Management -- Preventing Memory Overflow

AI models have a **context window** -- a maximum number of tokens they can process at once. If `agent.messages` gets too long, the model will reject the request.

**Conversation managers** handle this by removing old messages when the history gets too long.

| Manager | What it does |
|---------|-------------|
| `SlidingWindowConversationManager` | **Default.** Removes oldest messages when history exceeds token limit. |
| `NullConversationManager` | Does nothing. Messages grow forever (until error). |
| `SummarizingConversationManager` | Summarizes old messages instead of deleting them. |

**Source:** `src/strands/agent/conversation_manager/`

In [ ]:
# ============================================================
# STEP 4: SlidingWindowConversationManager
# ============================================================

from strands.agent.conversation_manager.sliding_window import SlidingWindowConversationManager

# Create a conversation manager with a small window.
# window_size controls how many messages to keep.
manager = SlidingWindowConversationManager()

print(f"Type: {type(manager).__name__}")
print()
print("This manager removes old messages when the conversation")
print("history exceeds the model's context window.")
print()
print("The agent uses this by default. You usually don't need")
print("to configure it unless you want custom behavior.")

In [ ]:
# ============================================================
# STEP 5: Watch messages grow with multiple calls
# ============================================================

# Create a simple agent and make multiple calls.
# Watch how the message count grows.

agent_multi = Agent(callback_handler=None)

questions = [
    "What is 1+1? Answer in one word.",
    "What is 2+2? Answer in one word.",
    "What is 3+3? Answer in one word.",
    "What is 4+4? Answer in one word.",
    "What is 5+5? Answer in one word.",
]

for q in questions:
    result = agent_multi(q)
    # After each call: 2 new messages (user + assistant)
    print(f"Messages: {len(agent_multi.messages):3d} | Q: {q[:30]} | A: {str(result)[:20]}")

print()
print(f"Final message count: {len(agent_multi.messages)}")
print("Each call adds 2 messages (user question + assistant answer).")
print("5 calls x 2 messages = 10 messages.")

In [ ]:
# ============================================================
# STEP 6: Show the message history
# ============================================================

print("=== Full Message History ===")
for i, msg in enumerate(agent_multi.messages):
    text = ""
    for block in msg['content']:
        if 'text' in block:
            text = block['text'][:60]
            break
    print(f"[{i:2d}] {msg['role']:10s} | {text}")

---
## NullConversationManager -- No Management

The `NullConversationManager` does nothing. Messages grow without limit. This is useful for short conversations or when you want full control.

In [ ]:
# ============================================================
# STEP 7: NullConversationManager comparison
# ============================================================

from strands.agent.conversation_manager.null import NullConversationManager

# NullConversationManager: does nothing, messages grow forever
agent_null = Agent(
    conversation_manager=NullConversationManager(),
    callback_handler=None,
)

# Make a few calls
for q in questions[:3]:
    result = agent_null(q)
    print(f"Messages: {len(agent_null.messages):3d} | Q: {q[:30]} | A: {str(result)[:20]}")

print()
print(f"With NullConversationManager: {len(agent_null.messages)} messages (nothing removed)")
print("This is fine for short conversations but will eventually")
print("hit the model's context window limit for long ones.")

---
## Summary

What you learned in this notebook:

- **Messages** are `{"role": "user"/"assistant", "content": [ContentBlock, ...]}` dictionaries
- **ContentBlock types:** text, toolUse, toolResult, image
- **toolUseId** links a toolUse request to its toolResult response
- **System prompt** is separate from messages -- sent to the model but not stored in `agent.messages`
- **Conversation managers** prevent context window overflow:
  - `SlidingWindowConversationManager` (default) removes old messages
  - `NullConversationManager` does nothing
  - `SummarizingConversationManager` summarizes old messages
- Each agent call adds 2 messages (no tools) or 4+ messages (with tools)

**Key source files:**
| File | What it does |
|------|--------------|
| `src/strands/types/content.py` | Message, ContentBlock types |
| `src/strands/agent/conversation_manager/` | All conversation managers |

**Next:** [NB7_Full_Architecture.ipynb](./NB7_Full_Architecture.ipynb) -- Everything together: multi-tool agent with hooks and conversation management